# Let's go PRO!

Advanced RAG Techniques!

Let's start by digging into ingest:

1. No LangChain! Just native for maximum flexibility
2. Let's use an LLM to divide up chunks in a sensible way
3. Let's use the best chunk size and encoder from yesterday
4. Let's also have the LLM rewrite chunks in a way that's most useful ("document pre-processing")

In [1]:
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go

import os


load_dotenv(override=True)

# MODEL = "gpt-4.1-nano"
# MODEL = "openrouter/nex-agi/nex-n2.5-mini:free"
MODEL = "openai/unsloth/gemma-4-E4B-it-GGUF:Q4_K_M"
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
API_KEY = ""

# BASE_URL = "https://openrouter.ai/api/v1"
BASE_URL = "http://127.0.0.1:8080/"

DB_NAME = "preprocessed_db"
collection_name = "docs"
embedding_model = "text-embedding-3-large"
# embedding_model = "all-MiniLM-L6-v2"
KNOWLEDGE_BASE_PATH = Path("knowledge-base")
AVERAGE_CHUNK_SIZE = 500

# embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

openai = OpenAI(
    # api_key = "",
    # base_url = BASE_URL
)

In [2]:
# Inspired by LangChain's Document - let's have something similar

class Result(BaseModel):
    page_content: str
    metadata: dict

In [3]:
# A class to perfectly represent a chunk

class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,metadata=metadata)


class Chunks(BaseModel):
    chunks: list[Chunk]

## Three steps:

1. Fetch documents from the knowledge base, like LangChain did
2. Call an LLM to turn documents into Chunks
3. Store the Chunks in Chroma

That's it!

### Let's start with Step 1

In [4]:
def fetch_documents():
    """A homemade version of the LangChain DirectoryLoader"""

    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})

    print(f"Loaded {len(documents)} documents")
    return documents

In [5]:
documents = fetch_documents()

Loaded 76 documents


In [6]:
documents[0]

{'type': 'company',
 'source': 'knowledge-base/company/about.md',
 'text': "# About Insurellm\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.\n\nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.\n\nHowever, the company underwent a strategic restructuring in 2022-2023 to focus on profitability and sustainable growth. This included consolidating office locations, implementing a remote-first strategy, and streamlining operations. As of 2025, Insurellm operates with a lean, highly efficient team of 32 employees who have built a portfolio of 32 active cont

### Donezo! On to Step 2 - make the chunks

In [7]:
def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""

In [8]:
print(make_prompt(documents[0]))


You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: company
The document has been retrieved from: knowledge-base/company/about.md

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into 5 chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

# About Insurellm

Insurellm was founded by Avery La

In [9]:
documents[0]

{'type': 'company',
 'source': 'knowledge-base/company/about.md',
 'text': "# About Insurellm\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.\n\nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.\n\nHowever, the company underwent a strategic restructuring in 2022-2023 to focus on profitability and sustainable growth. This included consolidating office locations, implementing a remote-first strategy, and streamlining operations. As of 2025, Insurellm operates with a lean, highly efficient team of 32 employees who have built a portfolio of 32 active cont

In [10]:
def make_messages(document):
    return [
        {"role": "user", "content": make_prompt(document)},
    ]

In [11]:
make_messages(documents[0])

[{'role': 'user',
  'content': "\nYou take a document and you split the document into overlapping chunks for a KnowledgeBase.\n\nThe document is from the shared drive of a company called Insurellm.\nThe document is of type: company\nThe document has been retrieved from: knowledge-base/company/about.md\n\nA chatbot will use these chunks to answer questions about the company.\nYou should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.\nThis document should probably be split into 5 chunks, but you can have more or less as appropriate.\nThere should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.\n\nFor each chunk, you should provide a headline, a summary, and the original text of the chunk.\nTogether your chunks should represent the entire document with overlap.\n\nHere is the document:\n\n# A

In [12]:
def process_document(document):
    messages = make_messages(document)
    response = completion(model=MODEL, messages=messages, response_format=Chunks, api_base=BASE_URL)
    reply = response.choices[0].message.content
    print(reply)
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]

In [13]:
# def process_document(document):
#     messages = [
#         {
#             "role": "system",
#             "content": (
#                 "Return only valid JSON with this structure: "
#                 '{"chunks": [{"headline": "...", "summary": "...", "original_text": "..."}]}'
#             ),
#         },
#         {"role": "user", "content": make_prompt(document)},
#     ]
#     response = completion(model=MODEL, messages=messages, response_format=Chunks)
#     reply = response.choices[0].message.content
#     print(reply)
#     doc_as_chunks = Chunks.model_validate_json(reply).chunks
#     return [chunk.as_result(document) for chunk in doc_as_chunks]

In [14]:
process_document(documents[0])

{
  "chunks": [
    {
      "headline": "Insurellm Founding and Early Growth",
      "summary": "Insurellm was founded in 2015 by Avery Lancaster as an insurance tech startup. It began with Markellm and quickly expanded its product offerings to include Carllm, Homellm, and Rellm.",
      "original_text": "# About Insurellm\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.\n\nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform)."
    },
    {
      "headline": "Peak Growth and Strategic Restructuring",
      "summary": "The company peaked in 2020 with 200 employees and 12 offices. In 2022-2023, Insurellm underwent a strategic restructuring to

[Result(page_content='Insurellm Founding and Early Growth\n\nInsurellm was founded in 2015 by Avery Lancaster as an insurance tech startup. It began with Markellm and quickly expanded its product offerings to include Carllm, Homellm, and Rellm.\n\n# About Insurellm\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.\n\nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform).', metadata={'source': 'knowledge-base/company/about.md', 'type': 'company'}),
 Result(page_content='Peak Growth and Strategic Restructuring\n\nThe company peaked in 2020 with 200 employees and 12 offices. In 2022-2023, Insurellm underwent a strategic restructuring to focus on

In [15]:
def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents):
        chunks.extend(process_document(doc))
    return chunks

In [16]:
chunks = create_chunks(documents)

  1%|▏         | 1/76 [00:25<32:02, 25.63s/it]

{
  "chunks": [
    {
      "headline": "Insurellm: Founding and Initial Growth (2015 - 2020)",
      "summary": "Insurellm was founded in 2015 to disrupt the insurance industry. It quickly expanded its product offering from an initial marketplace to include auto, home, and reinsurance platforms by 2020.",
      "original_text": "# About Insurellm\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.\n\nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US."
    },
    {
      "headline": "Strategic Restructuring and Operational Shift (2022-2025)",
      "summar

  3%|▎         | 2/76 [01:08<44:17, 35.91s/it]

{
  "chunks": [
    {
      "headline": "Insurellm Overview and Growth Journey",
      "summary": "Insurellm, founded in 2015, is a profitable, remote-first company that specializes in leveraging AI and automation for high-value enterprise clients in the insurance sector.",
      "original_text": "# Careers at Insurellm\n\n## Why Join Insurellm?\n\nAt Insurellm, we're not just building software—we're revolutionizing an entire industry. Since our founding in 2015, we've evolved from a high-growth startup to a lean, profitable company with 32 highly talented employees managing 32 active contracts across all eight of our product lines.\n\nAfter reaching 200 employees in 2020, we strategically restructured in 2022-2023 to focus on sustainable growth, operational excellence, and building a world-class remote-first culture. Today, we're a tight-knit team of exceptional professionals who deliver outsized impact through automation, AI, and strategic focus on high-value enterprise clients—from 

  4%|▍         | 3/76 [01:33<37:34, 30.88s/it]

...

{
  "chunks": [
    {
      "headline": "Lifellm Overview and Core Value Proposition",
      "summary": "Lifellm is an advanced AI-powered life insurance platform developed by Insurellm, designed to modernize how insurance providers offer coverage. It uses predictive analytics and streamlined processes to enhance customer experience and reduce costs for both B2B and B2C markets.",
      "original_text": "# Product Summary\n# Lifellm\n## Summary\nLifellm is an advanced life insurance platform developed by Insurellm that revolutionizes how insurance providers offer life coverage to their customers. Leveraging cutting-edge AI and predictive analytics, Lifellm enables insurers to provide personalized life insurance policies with streamlined underwriting, accurate risk assessment, and exceptional customer experiences. Designed for both B2B and B2C markets, Lifellm helps insurance companies modernize their life insurance operations while reducing costs and improving customer satisfaction.\n\

 99%|█████████▊| 75/76 [43:32<00:29, 29.20s/it]

{
  "chunks": [
    {
      "headline": "Markellm Product Overview: Connecting Consumers and Insurers",
      "summary": "Markellm is an innovative two-sided marketplace that connects consumers and insurance companies. It uses advanced AI matching to make the insurance shopping experience more efficient and personalized.",
      "original_text": "# Product Summary\n\n# Markellm\n\n## Summary\n\nMarkellm is an innovative two-sided marketplace designed to seamlessly connect consumers with insurance companies. Powered by advanced matching AI, Markellm transforms the insurance shopping experience, making it more efficient, personalized, and accessible. Whether you're a homeowner searching for the best rates on home insurance or an insurer looking to reach new customers, Markellm acts as the ultimate bridge, delivering tailored solutions for all parties involved. With a user-friendly interface and powerful algorithms, Markellm not only saves time but also enhances decision-making in the oft

100%|██████████| 76/76 [44:01<00:00, 34.76s/it]

{
  "chunks": [
    {
      "headline": "Rellm Overview: AI-Powered Reinsurance Transformation",
      "summary": "Rellm is an innovative enterprise reinsurance platform by Insurellm that uses AI to redefine risk management and optimize operational efficiencies for reinsurance companies.",
      "original_text": "# Product Summary\n\n# Rellm: AI-Powered Enterprise Reinsurance Solution\n\n## Summary\n\nRellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. Harnessing the power of artificial intelligence, Rellm offers an advanced platform that redefines risk management, enhances decision-making processes, and optimizes operational efficiencies within the reinsurance industry. With seamless integrations and robust analytics, Rellm enables insurers to proactively manage their portfolios and respond to market dynamics with agility.\n\n## Features\n\n### AI-Driven Analytics\nRellm utilizes cutting-edge AI alg

In [17]:
print(len(chunks))

556


### Well that was easy! If a bit slow.

In the python module version, I sneakily use the multi-processing Pool to run this in parallel,
but if you get a Rate Limit Error you can turn this off in the code.

### Finally, Step 3 - save the embeddings

In [18]:
def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]
    emb = openai.embeddings.create(model=embedding_model, input=texts).data
    vectors = [e.embedding for e in emb]

    collection = chroma.get_or_create_collection(collection_name)

    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"Vectorstore created with {collection.count()} documents")

In [19]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [20]:
def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)

    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]

    # Generate local Hugging Face embeddings
    vectors = embeddings.embed_documents(texts)

    collection = chroma.get_or_create_collection(collection_name)

    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(
        ids=ids,
        embeddings=vectors,
        documents=texts,
        metadatas=metas,
    )

    print(f"Vectorstore created with {collection.count()} documents")

In [21]:
create_embeddings(chunks)

Vectorstore created with 556 documents


# Nothing more to do here... right?

Wait! Didja think I'd forget??

In [22]:
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [23]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [24]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

## And now - let's build an Advanced RAG!

We will use these techniques:

1. Reranking - reorder the rank results
2. Query re-writing

In [25]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )

In [26]:
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = completion(model=MODEL, messages=messages, response_format=RankOrder, api_base=BASE_URL)
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    print(order)
    return [chunks[i - 1] for i in order]

In [27]:
# RETRIEVAL_K = 10

# def fetch_context_unranked(question):
#     query = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
#     results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
#     chunks = []
#     for result in zip(results["documents"][0], results["metadatas"][0]):
#         chunks.append(Result(page_content=result[0], metadata=result[1]))
#     return chunks

In [28]:
RETRIEVAL_K = 10

def fetch_context_unranked(question):
    query = embeddings.embed_query(question)
    results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
    chunks = []
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks

In [29]:
question = "Who won the IIOTY award?"
chunks = fetch_context_unranked(question)

In [30]:
for chunk in chunks:
    print(chunk.page_content[:15]+"...")

Awards, Trainin...
HR Development ...
Recent Performa...
Annual Performa...
Annual Performa...
Career Progress...
Annual Performa...
Skills, Develop...
James Wilson: A...
Annual Performa...


In [31]:
reranked = rerank(question, chunks)

[2, 1, 3, 4, 5, 6, 7, 8, 9, 10]


In [32]:
for chunk in reranked:
    print(chunk.page_content[:15]+"...")

HR Development ...
Awards, Trainin...
Recent Performa...
Annual Performa...
Annual Performa...
Career Progress...
Annual Performa...
Skills, Develop...
James Wilson: A...
Annual Performa...


In [33]:
question = "Who went to Manchester University?"
RETRIEVAL_K = 20
chunks = fetch_context_unranked(question)
for index, c in enumerate(chunks):
    if "manchester" in c.page_content.lower():
        print(index)

0


In [34]:
reranked = rerank(question, chunks)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


In [35]:
for index, c in enumerate(reranked):
    if "manchester" in c.page_content.lower():
        print(index)

0


In [36]:
reranked[0].page_content

"Performance Ratings and Compensation History (2020-2023)\n\nThis chunk covers the detailed performance ratings for 2020 and includes the entire history of Jessica Liu's compensation, showing increases in both base salary and bonus over the years.\n\n\n- **2020:** Rating: 3.8/5\n  *Good start as Junior Developer. Eager to learn and receptive to feedback.*\n\n## Compensation History\n- **2023:** Base Salary: $92,000 + Bonus: $6,000\n- **2022:** Base Salary: $85,000 + Bonus: $4,000\n- **2021:** Base Salary: $72,000 + Bonus: $2,000\n- **2020:** Base Salary: $68,000\n\n## Other HR Notes\n- **Education:** BS in Computer Science from University of Manchester\n- **Skills:** Proficient in React, TypeScript, HTML/CSS, Jest for testing. Learning Next.js and GraphQL.\n"

In [37]:
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

In [38]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

In [39]:
# In the context, include the source of the chunk

def make_rag_messages(question, history, chunks):
    context = "\n\n".join(f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

In [40]:
def rewrite_query(question, history=[]):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    message = f"""
You are in a conversation with a user, answering questions about the company Insurellm.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a single, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
Don't mention the company name unless it's a general question about the company.
IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""
    response = completion(model=MODEL, messages=[{"role": "system", "content": message}], api_base=BASE_URL)
    return response.choices[0].message.content

In [41]:
rewrite_query("Who won the IIOTY award?", [])

'IIOTY award winner'

In [42]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    """
    Answer a question using RAG and return the answer and the retrieved context
    """
    query = rewrite_query(question, history)
    print(query)
    chunks = fetch_context(query)
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=MODEL, messages=messages, api_base=BASE_URL)
    return response.choices[0].message.content, chunks

In [43]:
answer_question("Who won the IIOTY award?", [])

IIOTY award winner
[18, 1, 17, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 19, 20]


('Maxine Thompson won the prestigious Insurellm Innovator of the Year (IIOTY) 2023 award.',
 [Result(page_content="Transition to Data Engineer and Early Performance History (2018-2020)\n\nDetails Maxine's progression from Junior Data Engineer to Data Engineer (Nov 2018 - Dec 2020), noting communication challenges. It also covers performance reviews from 2018 to 2020.\n\n- **November 2018 - December 2020**: **Data Engineer**  \n  * In her new role, Maxine expanded her responsibilities to include designing comprehensive data models and improving data quality measures. Though she excelled in technical skills, communication issues with non-technical teams led to some project delays.\n- **January 2021 - Present**: **Senior Data Engineer**  \n  * Maxine was promoted to Senior Data Engineer after successfully leading a pivotal project that improved data retrieval times by 30%. She now mentors junior engineers and is involved in strategic data initiatives, solidifying her position as a valued 

In [44]:
answer_question("Who went to Manchester University?", [])

Manchester University graduates
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


('Jessica Liu attended the University of Manchester, where she received a BS in Computer Science.',
 [Result(page_content="Performance Ratings and Compensation History (2020-2023)\n\nThis chunk covers the detailed performance ratings for 2020 and includes the entire history of Jessica Liu's compensation, showing increases in both base salary and bonus over the years.\n\n\n- **2020:** Rating: 3.8/5\n  *Good start as Junior Developer. Eager to learn and receptive to feedback.*\n\n## Compensation History\n- **2023:** Base Salary: $92,000 + Bonus: $6,000\n- **2022:** Base Salary: $85,000 + Bonus: $4,000\n- **2021:** Base Salary: $72,000 + Bonus: $2,000\n- **2020:** Base Salary: $68,000\n\n## Other HR Notes\n- **Education:** BS in Computer Science from University of Manchester\n- **Skills:** Proficient in React, TypeScript, HTML/CSS, Jest for testing. Learning Next.js and GraphQL.\n", metadata={'source': 'knowledge-base/employees/Jessica Liu.md', 'type': 'employees'}),
  Result(page_content

In [45]:
answer_question("Tell me about the healthllm product", [])

healthllm product
[2, 9, 4, 17, 14, 15, 16, 11, 18, 7, 1, 6, 5, 3, 10, 13, 12, 19, 8, 20]


("Hello! I would be delighted to tell you about Healthllm.\n\n**Healthllm** is Insurellm's comprehensive, AI-powered platform specifically designed for the health insurance industry. Its core mission is to streamline every facet of health insurance operations, helping providers enhance member outcomes, significantly reduce costs, and navigate the complex regulatory landscape with confidence.\n\nHere is a detailed overview of the product's capabilities, features, and pricing:\n\n### 🧬 Core Functionality & AI Features\n\nHealthllm uses advanced AI and healthcare data analytics to automate and optimize processes across the entire value chain:\n\n*   **Intelligent Plan Design:** The platform helps insurers create competitive health plans by analyzing market trends, member demographics, and healthcare utilization patterns, suggesting optimal benefit structures, deductibles, and network configurations.\n*   **Real-Time Eligibility Verification:** It instantly confirms a patient's coverage st